# Spark

## Example

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import monotonically_increasing_id

In [2]:
spark = SparkSession \
            .builder \
            .appName("test") \
            .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/24 23:13:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
df = spark.read.csv("./people.csv", header=True, sep=';')
df.show()

+-----+---+---------+
| name|age|      job|
+-----+---+---------+
|Jorge| 30|Developer|
|  Bob| 32|Developer|
+-----+---+---------+



In [6]:
df.count()

2

In [7]:
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- job: string (nullable = true)



In [8]:
df.select("name").show()
df.select(["name", "job"]).show()

+-----+
| name|
+-----+
|Jorge|
|  Bob|
+-----+

+-----+---------+
| name|      job|
+-----+---------+
|Jorge|Developer|
|  Bob|Developer|
+-----+---------+



In [9]:
df.filter(df['age'] > 31).show()

+----+---+---------+
|name|age|      job|
+----+---+---------+
| Bob| 32|Developer|
+----+---+---------+



In [10]:
df.withColumn('index', monotonically_increasing_id())

DataFrame[name: string, age: string, job: string, index: bigint]

## Learning on massive click logs with Spark

In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructField, StringType, StructType, IntegerType
from pyspark.ml.feature import StringIndexer, VectorAssembler, OneHotEncoder
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [13]:
spark = SparkSession\
    .builder\
    .appName("CTR")\
    .getOrCreate()

26/06/24 23:16:26 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [14]:
schema = StructType([
    StructField("id", StringType(), True),
    StructField("click", IntegerType(), True),
    StructField("hour", IntegerType(), True),
    StructField("C1", StringType(), True),
    StructField("banner_pos", StringType(), True),
    StructField("site_id", StringType(), True),
    StructField("site_domain", StringType(), True),
    StructField("site_category", StringType(), True),
    StructField("app_id", StringType(), True),
    StructField("app_domain", StringType(), True),
    StructField("app_category", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("device_ip", StringType(), True),
    StructField("device_model", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("device_conn_type", StringType(), True),
    StructField("C14", StringType(), True),
    StructField("C15", StringType(), True),
    StructField("C16", StringType(), True),
    StructField("C17", StringType(), True),
    StructField("C18", StringType(), True),
    StructField("C19", StringType(), True),
    StructField("C20", StringType(), True),
    StructField("C21", StringType(), True),
])

In [15]:
df = spark.read.csv("file:///Users/borjadh/Desktop/Borja/Python-Machine-Learning-By-Example-Third-Edition/chapter6/train", schema=schema, header=True)

26/06/24 23:16:31 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: file:///Users/borjadh/Desktop/Borja/Python-Machine-Learning-By-Example-Third-Edition/chapter6/train.
java.io.FileNotFoundException: File file:/Users/borjadh/Desktop/Borja/Python-Machine-Learning-By-Example-Third-Edition/chapter6/train does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSour

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/Users/borjadh/Desktop/Borja/Python-Machine-Learning-By-Example-Third-Edition/chapter6/train. SQLSTATE: 42K03

In [ ]:
df.printSchema()

In [ ]:
df.count()

In [ ]:
df = df.drop('id').drop('hour').drop('device_id').drop('device_ip')
df = df.withColumnRenamed("click", "label")

In [ ]:
df.columns

In [ ]:
df_train, df_test = df.randomSplit([0.7, 0.3], 42)

In [ ]:
df_train.cache()

In [ ]:
df_train.count()

In [ ]:
df_test.cache()

In [ ]:
df_test.count()

### One-hot encoding categorical features

In [ ]:
categorical = df_train.columns
categorical.remove('label')
print(categorical)

In [ ]:
indexers = [
    StringIndexer(inputCol=c, outputCol="{0}_indexed".format(c)).setHandleInvalid("keep")
    for c in categorical
]

In [ ]:
indexers

In [ ]:
encoder = OneHotEncoder(
    inputCols=[indexer.getOutputCol() for indexer in indexers],
    outputCols=[
        "{0}_encoded".format(indexer.getOutputCol()) for indexer in indexers]
)
assembler = VectorAssembler(
    inputCols=encoder.getOutputCols(),
    outputCol="features"
)

In [ ]:
stages = indexers + [encoder, assembler]
pipeline = Pipeline(stages=stages)


In [ ]:
one_hot_encoder = pipeline.fit(df_train)

In [ ]:
df_train_encoded = one_hot_encoder.transform(df_train)
df_train_encoded.show()

In [ ]:
df_train_encoded = df_train_encoded.select(["label", "features"])
df_train_encoded.show()

In [ ]:
df_train_encoded.cache()

In [ ]:
df_train.unpersist()

In [ ]:
df_test_encoded = one_hot_encoder.transform(df_test)
df_test_encoded = df_test_encoded.select(["label", "features"])
df_test_encoded.show()

In [ ]:
df_test_encoded.cache()

In [ ]:
df_test.unpersist()

### Training and testing a logistic regression model

In [ ]:
classifier = LogisticRegression(maxIter=20, regParam=0.000, elasticNetParam=0.000)

In [ ]:
lr_model = classifier.fit(df_train_encoded)

In [ ]:
df_train_encoded.unpersist()

In [ ]:
predictions = lr_model.transform(df_test_encoded)

In [ ]:
df_test_encoded.unpersist()

In [ ]:
predictions.cache()

In [ ]:
predictions.show()

In [ ]:
ev = BinaryClassificationEvaluator(rawPredictionCol = "rawPrediction", metricName = "areaUnderROC")

In [ ]:
print(ev.evaluate(predictions))

In [ ]:
spark.stop()

## Feature engineering on categorical variables with Spark

### Feature hashing

In [ ]:
from pyspark.ml.feature import FeatureHasher

In [ ]:
spark = SparkSession\
    .builder\
    .appName("CTR")\
    .getOrCreate()

In [ ]:
schema = StructType([
    StructField("id", StringType(), True),
    StructField("click", IntegerType(), True),
    StructField("hour", IntegerType(), True),
    StructField("C1", StringType(), True),
    StructField("banner_pos", StringType(), True),
    StructField("site_id", StringType(), True),
    StructField("site_domain", StringType(), True),
    StructField("site_category", StringType(), True),
    StructField("app_id", StringType(), True),
    StructField("app_domain", StringType(), True),
    StructField("app_category", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("device_ip", StringType(), True),
    StructField("device_model", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("device_conn_type", StringType(), True),
    StructField("C14", StringType(), True),
    StructField("C15", StringType(), True),
    StructField("C16", StringType(), True),
    StructField("C17", StringType(), True),
    StructField("C18", StringType(), True),
    StructField("C19", StringType(), True),
    StructField("C20", StringType(), True),
    StructField("C21", StringType(), True),
])

In [ ]:
df = spark.read.csv("file:///Users/borjadh/Desktop/Borja/Python-Machine-Learning-By-Example-Third-Edition/chapter6/train", schema=schema, header=True)

In [ ]:
df = df.drop('id').drop('hour').drop('device_id').drop('device_ip')
df = df.withColumnRenamed("click", "label")
df_train, df_test = df.randomSplit([0.7, 0.3], 42)
df_train.cache()
df_test.cache()

In [ ]:
categorical = df_train.columns
categorical.remove('label')
print(categorical)

In [ ]:
hasher = FeatureHasher(numFeatures=10000, inputCols=categorical,
                       outputCol="features")

In [ ]:
hasher.transform(df_train).select("features").show()

In [ ]:
classifier = LogisticRegression(maxIter=20, regParam=0.000, elasticNetParam=0.000)

In [ ]:
stages = [hasher, classifier]
pipeline = Pipeline(stages=stages)

In [ ]:
model = pipeline.fit(df_train)

In [ ]:
predictions = model.transform(df_test)

In [ ]:
predictions.cache()

In [ ]:
ev = BinaryClassificationEvaluator(rawPredictionCol = "rawPrediction", metricName = "areaUnderROC")

In [ ]:
print(ev.evaluate(predictions))

In [ ]:
spark.stop()

### Feature interaction

In [ ]:
from pyspark.ml.feature import RFormula

In [ ]:
spark = SparkSession\
    .builder\
    .appName("CTR")\
    .getOrCreate()

In [ ]:
schema = StructType([
    StructField("id", StringType(), True),
    StructField("click", IntegerType(), True),
    StructField("hour", IntegerType(), True),
    StructField("C1", StringType(), True),
    StructField("banner_pos", StringType(), True),
    StructField("site_id", StringType(), True),
    StructField("site_domain", StringType(), True),
    StructField("site_category", StringType(), True),
    StructField("app_id", StringType(), True),
    StructField("app_domain", StringType(), True),
    StructField("app_category", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("device_ip", StringType(), True),
    StructField("device_model", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("device_conn_type", StringType(), True),
    StructField("C14", StringType(), True),
    StructField("C15", StringType(), True),
    StructField("C16", StringType(), True),
    StructField("C17", StringType(), True),
    StructField("C18", StringType(), True),
    StructField("C19", StringType(), True),
    StructField("C20", StringType(), True),
    StructField("C21", StringType(), True),
])

In [ ]:
df = spark.read.csv("file:///Users/borjadh/Desktop/Borja/Python-Machine-Learning-By-Example-Third-Edition/chapter6/train", schema=schema, header=True)
df = df.drop('id').drop('hour').drop('device_id').drop('device_ip')
df = df.withColumnRenamed("click", "label")
df_train, df_test = df.randomSplit([0.7, 0.3], 42)
df_train.cache()
df_test.cache()

In [ ]:
categorical = df_train.columns
categorical.remove('label')
print(categorical)

In [ ]:
cat_inter = ['C14', 'C15']
concat = '+'.join(categorical)
interaction = ':'.join(cat_inter)
formula = "label ~ " + concat + '+' + interaction

In [ ]:
print(formula)

In [ ]:
interactor = RFormula(
    formula=formula,
    featuresCol="features",
    labelCol="label").setHandleInvalid("keep")

In [ ]:
interactor.fit(df_train).transform(df_train).select("features").show()

In [ ]:
classifier = LogisticRegression(maxIter=20, regParam=0.000, elasticNetParam=0.000)

In [ ]:
stages = [interactor, classifier]
pipeline = Pipeline(stages=stages)
model = pipeline.fit(df_train)
predictions = model.transform(df_test)
predictions.cache()
predictions.show()

In [ ]:
ev = BinaryClassificationEvaluator(rawPredictionCol = "rawPrediction", metricName = "areaUnderROC")

In [ ]:
print(ev.evaluate(predictions))

In [ ]:
spark.stop()